### CÀI ĐẶT THƯ VIỆN

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
from scipy import stats

# Dataset mặc định Iris
df = sns.load_dataset("iris")

---

### Phần 1: Thống kê mô tả & đặc trưng

In [ ]:
# 1. Đọc dữ liệu, hiển thị 5 dòng đầu
print("5 dòng đầu tiên của dữ liệu:")
print(df.head())
print(f"\nSố dòng: {df.shape[0]}, Số cột: {df.shape[1]}")

# 2. Thống kê mô tả cho từng biến số
numeric_cols = df.select_dtypes(include=[np.number]).columns
stats_df = pd.DataFrame({
    'mean': df[numeric_cols].mean(),
    'median': df[numeric_cols].median(),
    'mode': df[numeric_cols].mode().iloc[0],
    'var': df[numeric_cols].var(),
    'std': df[numeric_cols].std(),
    'min': df[numeric_cols].min(),
    'max': df[numeric_cols].max(),
    'Q1': df[numeric_cols].quantile(0.25),
    'Q3': df[numeric_cols].quantile(0.75),
})
stats_df['IQR'] = stats_df['Q3'] - stats_df['Q1']
print("\nBảng thống kê mô tả:")
display(stats_df) # Dùng display() trong notebook để bảng đẹp hơn

# 3. Tính mean và std theo từng nhóm species
group_stats = df.groupby('species').agg(['mean', 'std'])
print("\nThống kê theo loài (species):")
display(group_stats)

#### Nhận xét:
* Dữ liệu Iris gồm 150 dòng và 5 cột trong đó có 4 biến số, 1 biến phân loại.
* Khi nhóm theo loài (species), thuộc tính petal_length và petal_width cho thấy sự khác biệt rõ rệt nhất về giá trị trung bình (mean) giữa 3 loài Setosa, Versicolor và Virginica.

---

### Phần 2: Phân phối xác suất

In [ ]:
# 1. Vẽ histogram + KDE cho từng biến số
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i//2, i%2])
    axes[i//2, i%2].set_title(f'Histogram & KDE của {col}')
plt.tight_layout()
plt.show()

# 2. Vẽ boxplot từng biến theo nhóm
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, col in enumerate(numeric_cols):
    sns.boxplot(x='species', y=col, data=df, ax=axes[i//2, i%2])
    axes[i//2, i%2].set_title(f'Boxplot của {col} theo species')
plt.tight_layout()
plt.show()

# 3. Mô phỏng với biến sepal_width
col_sim = 'sepal_width'
mean_val = df[col_sim].mean()
std_val = df[col_sim].std()

plt.figure(figsize=(8, 6))
# Vẽ histogram thực tế
sns.histplot(df[col_sim], stat='density', alpha=0.5, label='Dữ liệu thực tế')

# Vẽ đường Normal PDF lý thuyết
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = stats.norm.pdf(x, mean_val, std_val)
plt.plot(x, p, 'k', linewidth=2, label='PDF Lý thuyết')
plt.title(f'Mô phỏng phân phối Normal cho {col_sim}')
plt.legend()
plt.show()

#### Nhận xét:
* Hình dạng phân phối:
    * sepal_width và sepal_length có phân phối khá gần với phân phối chuẩn (Normal Distribution).
    * petal_length và petal_width có dạng đa đỉnh (bimodal) do sự phân tách rõ rệt của loài Setosa so với 2 loài còn lại.
* Mức độ khớp: Khi áp đường PDF lý thuyết lên histogram của sepal_width, đường cong ôm khá sát vào phân phối thực tế của dữ liệu, chứng tỏ biến này gần như tuân theo phân phối chuẩn. 

---

### Phần 3: Phân tích đa biến và tương quan

In [ ]:
# 1. Tính ma trận hiệp phương sai và tương quan
cov_matrix = df[numeric_cols].cov()
corr_matrix = df[numeric_cols].corr()

print("Ma trận hiệp phương sai (Covariance):")
display(cov_matrix)
print("\nMa trận tương quan (Correlation):")
display(corr_matrix)

# 2. Vẽ heatmap tương quan
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Heatmap Tương Quan')
plt.show()

# 3. Vẽ pairplot
sns.pairplot(df, hue='species', palette='Set1')
plt.suptitle('Pairplot của dataset Iris', y=1.02)
plt.show()

#### Nhận xét: 
* Cặp biến tương quan mạnh nhất là petal_length và petal_width với hệ số tương quan dương rất cao $(0.96)$. Vì 2 biến này có tương quan tuyến tính gần như hoàn hảo do đó có sự hiện diện của hiện tượng đa cộng tuyến.
* Trên đồ thị pairplot, loài Setosa (màu đỏ) luôn nằm tách biệt hẳn so với Versicolor và Virginica trên các trục liên quan đến "petal".

---

### Phần 4: Xác suất và định lý Bayes

In [ ]:
# 1. Tính xác suất hậu nghiệm P(B|+)
P_B = 0.01          # Tỉ lệ mắc bệnh
P_pos_given_B = 0.99   # Độ nhạy (True positive)
P_pos_given_NB = 0.05  # Dương tính giả (False positive)
P_NB = 1 - P_B         # Tỉ lệ không mắc bệnh

# Xác suất xét nghiệm dương tính P(+)
P_pos = (P_pos_given_B * P_B) + (P_pos_given_NB * P_NB) # P(+) = P(+|B) * P(B) + P(+|¬B) * P(¬B)

# Định lý Bayes
P_B_given_pos = (P_pos_given_B * P_B) / P_pos  # P(B|+) = (P(+|B) * P(B)) / P(+)

print(f"Xác suất thực sự mắc bệnh khi xét nghiệm dương tính là: {P_B_given_pos:.4f} (khoảng {P_B_given_pos*100:.2f}%)")

# 2. Vẽ đồ thị khảo sát P(B|+) khi P(B) thay đổi
P_B_array = np.linspace(0.001, 0.2, 200)
P_NB_array = 1 - P_B_array
P_pos_array = (P_pos_given_B * P_B_array) + (P_pos_given_NB * P_NB_array)
P_B_given_pos_array = (P_pos_given_B * P_B_array) / P_pos_array

plt.figure(figsize=(8, 5))
plt.plot(P_B_array, P_B_given_pos_array, color='b', linewidth=2)
plt.xlabel('Tỉ lệ mắc bệnh trong dân số: P(B)')
plt.ylabel('Xác suất hậu nghiệm: P(B|+)')
plt.title('Sự thay đổi của P(B|+) theo tỉ lệ mắc bệnh P(B)')
plt.grid(True)
plt.show()


#### Nhận xét: 
* Khi bệnh rất hiếm $P(B) = 0.01$, số lượng người khỏe mạnh đi xét nghiệm là cực kỳ lớn. Do đó, tỉ lệ $5\%$ dương tính giả của nhóm người khỏe mạnh này sẽ tạo ra một số lượng kết quả dương tính lớn áp đảo hoàn toàn so với số ca dương tính thật. Điều này kéo thấp giá trị dự đoán dương tính xuống.